# 1 load package and data

In [ ]:
import json
from collections import namedtuple, Counter

import numpy as np
import pandas as pd
import scanpy as sc
import pandas as pd
import squidpy as sq

## Image manipulation and geometry
from tifffile import imread
from skimage.io import imread as skimread

## Plotting imports
from matplotlib import pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize, to_hex, Colormap
from matplotlib.cm import ScalarMappable
from matplotlib.colorbar import ColorbarBase
from matplotlib.lines import Line2D
from matplotlib import rc_context

# 2.Extended Data Fig. 8a

In [ ]:
folder_path = r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\analysis\2st_2025_4_9\backup_adata\QC_10_50_2st"
h5ad_file = Path(folder_path) / "embryont_extra.h5ad"
adata = sc.read_h5ad(h5ad_file)

In [ ]:
my_genes=[
    "COL5A1","GJA1","RGS4","GDF7", "TWIST1","RSPO2","THBS1", "EGFL6", "APOB","POSTN","FOXA1","LOX","HBZ", "PLVAP","EPCAM","VTCN1","MYBL2", "CREG1","IGFBP2","AXIN2","FABP7", "CDX2","PDGFA"
]
sc.pl.dotplot(
    adata,
    var_names=my_genes,
    groupby='leiden_hvg_sub',  # 你的分组列名
    standard_scale='var',  # 标准化
    #vmax=2, 
    cmap="Reds",
    figsize=(8, 4),
    dendrogram=True,
    save='EE_leiden_sub_normalize.pdf'
)

# 2.Extended Data Fig. 8h

In [ ]:
import scanpy as sc
import pandas as pd
from pathlib import Path
# 文件路径
folder_path = r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\analysis\2st_2025_4_9\backup_adata\QC_10_50_2st\placenta"
h5ad_file = Path(folder_path) / "whole_placenta_updata_hvg_leiden_filter_3st_with_newcordination.h5ad"
placenta = sc.read_h5ad(h5ad_file)

In [ ]:
import squidpy as sq
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


adata =placenta   
axis_name = 'AP'  
axis_col = 'AP'  
n_neighs = 15     
n_perms = 100     
n_jobs = 4       
fdr_threshold = 0.05  
output_prefix = 'squidpy_moranI_DV' 


print("=" * 60)
print(f"对{axis_name}
print("=" * 60)


print(f"{axis_name}")
spatial_key = f'spatial_{axis_name}'
adata.obsm[spatial_key] = np.column_stack([
    adata.obs[axis_col].values,  
    np.zeros(len(adata.obs))      
])


print(f"{axis_name}（n_neighs={n_neighs}）...")
sq.gr.spatial_neighbors(
    adata,
    coord_type='generic',
    n_neighs=n_neighs,
    spatial_key=spatial_key,
    key_added=spatial_key
)


print(f"{axis_name}Moran's I（n_perms={n_perms}）...")
sq.gr.spatial_autocorr(
    adata,
    mode='moran',
    n_perms=n_perms,
    n_jobs=n_jobs,
    connectivity_key=f'{spatial_key}_connectivities'
)


results = adata.uns['moranI'].copy()
results = results.sort_values('I', ascending=False)

print(f"\n {len(results)} Moran's I ({axis_name})")
print(f" (FDR<{fdr_threshold}): {(results['pval_norm_fdr_bh'] < fdr_threshold).sum()}")
print(f"\nTop 20 {axis_name}:")
print(results.head(20)[['I', 'pval_norm', 'pval_norm_fdr_bh']])


csv_file = f'{output_prefix}_results_gut.csv'
results.to_csv(csv_file)




fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, gene in enumerate(results.head(6).index):
    ax = axes[i]
    

    gene_expr = adata[:, gene].X.toarray().ravel() if hasattr(adata[:, gene].X, 'toarray') else adata[:, gene].X.ravel()
    axis_coords = adata.obs[axis_col].values
    
 
    scatter = ax.scatter(axis_coords, 
                        np.random.randn(len(axis_coords)) * 0.1,  
                        c=gene_expr, 
                        s=2, 
                        cmap='viridis', 
                        alpha=0.6)
    
 
    moran_i = results.loc[gene, 'I']
    pval = results.loc[gene, 'pval_norm_fdr_bh']
    ax.set_title(f'{gene}\nMoran I={moran_i:.3f}, FDR={pval:.2e}', fontsize=9)
    ax.set_xlabel(f'{axis_name} Position')
    ax.set_ylabel('Random jitter')
    plt.colorbar(scatter, ax=ax, label='Expression')

plt.tight_layout()
png_file = f'{output_prefix}_top6_genes.png'
plt.savefig(png_file, dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
my_genes = [
   "TFAP2A","EPCAM","SOX9","ANXA2", "DLK1","H19", "EIF1B", "PAX1",
    "PITX2", "NKX2-1"
]

sc.pl.dotplot(
    c6,
    var_names=my_genes,
    groupby='leiden',  # 你的分组列名
    standard_scale='var',  # 标准化
    #vmax=2, 
    cmap="Reds",
    figsize=(5, 2.5),
    dendrogram=True,
    save='C6_leiden_sub_normalize.pdf'
)

In [ ]:
genes = results[results['I'] > 0.1].index.tolist()
genes_to_remove = ['AKR1B1', 'CDKN1C', 'GATA2', 'SMAD3', 'HBG1',"HBZ",'SLC4A1', 'SMAD3']
genes = [g for g in genes if g not in genes_to_remove]

In [ ]:
 norm_df = create_simple_comparison_heatmap(
   placenta, genes, axis_col='AP',
    n_bins=60, figsize=(5, 6),
    order_method='peak_grouped',  #spearman   peak
    binning='linear',#quantile   linear
    smooth='gaussian', smooth_sigma=2,
    normalize_genes=True,
    data_type='normalized',
    show_only_highlighted=True,     
    #show_highlight_marker=False, 
    vmin=-2, vmax=2,
    vlines=[0, 1500, 1900],
    vline_color='blue',
    vline_style='--',
    hlines=["MRC2","PKM","PEA15"],
    hline_color='black',
    hline_width=2,
    xlim=(-300, 2800),
    highlight_genes=["RELN",'ANGPT2','FLT4',"MAFB"],
    highlight_color='red',
    highlight_size=2,
    highlight_label_position='left',
    save_path="P:/PI/PI_Chuva_de_Sousa_Lopes/susana/Fu/Project/20_Xenium 5k embryo/paper/3st_2025_1_27/Figure6/along_axis/placenta_along_axis_AP.pdf"
     
)

# 3.Extended Data Fig. 8i

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import make_interp_spline
import matplotlib.ticker as mticker

# ============================================================
# Fully enhanced version: Plot gene expression along AP axis (supports subset filtering and dual Y-axes)
# ============================================================
def plot_genes_along_AP(
    adata,                          # NEW: AnnData object
    gene_list=None,                 # Gene name list (simple mode)
    gene_configs=None,              # NEW: Gene configuration list (advanced mode)
    coord_col='AP',                 # NEW: coordinate column name
    subset_col='sub_leiden',        # NEW: column used for subset filtering
    n_bins=50, 
    show_points=True, 
    smooth=0, 
    figsize=(12, 6), 
    linewidth=2, 
    fill=False,
    fill_alpha=0.3,                 # NEW: fill transparency
    gene_colors=None, 
    gene_linestyles=None,
    vertical_lines=None, 
    vline_color='blue',
    vline_style='--', 
    vline_width=1.5, 
    vline_alpha=0.8,
    show_legend=True,
    xlim=None,                      # NEW: X-axis limits, e.g. (0, 2500)
    ylim=None,                      # NEW: Primary (left) Y-axis limits, e.g. (0,1)
    x_tick_interval=100,            # NEW: X-axis tick interval
    secondary_ylim=None,            # NEW: Secondary (right) Y-axis limits
    normalization='minmax'          # NEW: normalization method: 'minmax', 'zscore', 'none'
):
    

    if gene_configs is None and gene_list is None:
        raise ValueError("Either gene_list or gene_configs must be provided")
    
    if coord_col not in adata.obs.columns:
        raise KeyError(f"Coordinate column not found in obs '{coord_col}'")
    if subset_col not in adata.obs.columns:
        raise KeyError(f"Subset column not found in obs '{subset_col}'")
    
    # 简单模式：Convert gene_list to gene_configs format
    if gene_configs is None:
        gene_configs = [{
            'genes': gene_list,
            'subset_values': None,  # Use all cells
            'use_secondary_y': False,
            'linestyle': None,  # use gene_linestyles
            'color': None       # use gene_colors
        }]
    
    # ========== 2. Create figure and axes ==========
    fig, ax = plt.subplots(figsize=figsize)
    
    # Check whether a secondary Y-axis is needed
    need_secondary_y = any(config.get('use_secondary_y', False) for config in gene_configs)
    ax2 = None
    if need_secondary_y:
        ax2 = ax.twinx()
    
    # ========== 3. Process each configuration group ==========
    all_gene_curves = {}
    
    for config in gene_configs:
        genes = config['genes']
        subset_values = config.get('subset_values', None)
        use_secondary_y = config.get('use_secondary_y', False)
        config_linestyle = config.get('linestyle', '-')
        config_color = config.get('color', None)
        config_n_bins = config.get('n_bins', n_bins)  # Using 配置的n_bins或全局n_bins
        
        # Select current Y-axis
        current_ax = ax2 if (use_secondary_y and ax2 is not None) else ax
        
        # Filter cell subset
        if subset_values is not None:
            # Convert subset_values to string list
            subset_values_str = [str(v) for v in subset_values]
            mask = adata.obs[subset_col].astype(str).isin(subset_values_str)
            adata_subset = adata[mask, :].copy()
            print(f"Using {subset_col}中值为{subset_values} cells, total {mask.sum()} cells, bins={config_n_bins}")
        else:
            adata_subset = adata
            print(f"Use all cells，共{adata_subset.n_obs} cells, bins={config_n_bins}")
        
        # Get and sort coordinate values
        coords = adata_subset.obs[coord_col].values
        order = np.argsort(coords)
        coords_sorted = coords[order]
  
        bins = np.linspace(coords_sorted.min(), coords_sorted.max(), config_n_bins+1)
        bin_indices = np.digitize(coords_sorted, bins) - 1
        bin_indices = np.clip(bin_indices, 0, config_n_bins-1)
        
        # Calculate bin centers
        bin_centers = (bins[:-1] + bins[1:]) / 2
        
        # ========== 4. Calculate expression curve for each gene ==========
        for gene in genes:
            if gene not in adata_subset.var_names:
                print(f"Warning: {gene}  not found, skipped")
                continue
            
            # Get expression
            expr = adata_subset[:, gene].X
            if hasattr(expr, 'toarray'):
                expr = expr.toarray().ravel()
            else:
                expr = expr.ravel()
            expr_sorted = expr[order]
            
            # Calculate mean expression for each bin
            bin_means = np.array([expr_sorted[bin_indices == i].mean() 
                                  if np.any(bin_indices == i) else 0 
                                  for i in range(config_n_bins)])  # Using config_n_bins
            
            # Normalization
            if normalization == 'minmax':
                bin_means_norm = (bin_means - bin_means.min()) / (bin_means.max() - bin_means.min() + 1e-10)
            elif normalization == 'zscore':
                bin_means_norm = (bin_means - bin_means.mean()) / (bin_means.std() + 1e-10)
            elif normalization == 'none':
                bin_means_norm = bin_means
            else:
                raise ValueError(f"未知的Normalization方法: {normalization}")
            
            all_gene_curves[gene] = bin_means_norm
            
            # ========== 5. Plot curves ==========
            # Determine color and linestyle
            if config_color:
                color = config_color
            elif gene_colors and gene in gene_colors:
                color = gene_colors[gene]
            else:
                color = None
            
            if config_linestyle:
                linestyle = config_linestyle
            elif gene_linestyles and gene in gene_linestyles:
                linestyle = gene_linestyles[gene]
            else:
                linestyle = '-'
            
            # Plot curves
            if smooth > 0:
                # Create dense x values for smoothing
                x_smooth = np.linspace(bin_centers.min(), bin_centers.max(), config_n_bins * 10)
                
                try:
                    k = min(int(smooth), len(bin_centers) - 1, 5)
                    spl = make_interp_spline(bin_centers, bin_means_norm, k=k)
                    y_smooth = spl(x_smooth)
                    
         
                    if normalization == 'minmax':
                        y_smooth = np.clip(y_smooth, 0, 1)
                    
                    # Fill area under curve
                    if fill:
                        current_ax.fill_between(x_smooth, y_smooth, alpha=fill_alpha, color=color)
                    
                    # Draw smoothed curve
                    current_ax.plot(x_smooth, y_smooth, linestyle=linestyle, label=gene, 
                                   linewidth=linewidth, color=color, alpha=0.7)
                    
                    # Show original data points
                    if show_points:
                        current_ax.plot(bin_centers, bin_means_norm, 'o', markersize=4, 
                                       alpha=0.5, color=color)
                except Exception as e:
                    print(f"Warning: {gene} Smoothing failed ({str(e)})，Using 原始曲线")
                    marker = 'o-' if show_points else '-'
                    
                    if fill:
                        current_ax.fill_between(bin_centers, bin_means_norm, alpha=fill_alpha, color=color)
                    
                    current_ax.plot(bin_centers, bin_means_norm, marker, label=gene, 
                                   linewidth=linewidth, color=color,
                                   linestyle=linestyle if not show_points else '-',
                                   markersize=4 if show_points else 0, alpha=0.7)
            else:
                # Without smoothing
                marker = 'o-' if show_points else '-'
                
                if fill:
                    current_ax.fill_between(bin_centers, bin_means_norm, alpha=fill_alpha, color=color)
                
                current_ax.plot(bin_centers, bin_means_norm, marker, label=gene, 
                               linewidth=linewidth, color=color,
                               linestyle=linestyle if not show_points else '-',
                               markersize=4 if show_points else 0, alpha=0.7)
    
    # ========== 6. Add vertical reference lines ==========
    if vertical_lines:
        for line_pos in vertical_lines:
            ax.axvline(x=line_pos, color=vline_color, linestyle=vline_style,
                      linewidth=vline_width, alpha=vline_alpha, zorder=10)
    
    # ========== 7. Configure axes ==========

    if xlim:
        ax.set_xlim(xlim)
    if ylim:
        ax.set_ylim(ylim)
    if secondary_ylim and ax2:
        ax2.set_ylim(secondary_ylim)
    
    ax.set_xlabel(f'{coord_col} Position', fontsize=12)
    
    ylabel_dict = {
        'minmax': 'Normalized Expression (0-1)',
        'zscore': 'Z-score Normalized Expression',
        'none': 'Expression Level'
    }
    ax.set_ylabel(ylabel_dict.get(normalization, 'Expression'), fontsize=12)
    
    if ax2:
        ax2.set_ylabel(f'{ylabel_dict.get(normalization, "Expression")} (Secondary)', fontsize=12)
    
    ax.set_title(f'Gene Expression along {coord_col} axis', fontsize=14, fontweight='bold')
    ax.xaxis.set_major_locator(mticker.MultipleLocator(x_tick_interval))
    
 
    if show_legend:
        lines1, labels1 = ax.get_legend_handles_labels()
        if ax2:
            lines2, labels2 = ax2.get_legend_handles_labels()
            ax.legend(lines1 + lines2, labels1 + labels2, 
                     bbox_to_anchor=(1.15, 1), loc='upper left')
        else:
            ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    ax.margins(x=0, y=0.02)
    plt.tight_layout()
    
    return fig, all_gene_curves

In [ ]:


vertical_lines = [0, 1500, 1900]
gene_colors = {
    "RELN":  "#FF0000",  # 0
    "ANGPT2":  "#00FF00",  # 1
    #"FLRT3":  "#FE664D",  # 2
    #"DIO2": "#565DFD",  # 7
    #"VGF": "#00FFFF",  # 8
    #"HOXB4": "#1068be",  # 10
    #"TRPA1": "#565DFD",  # 11
    #"GLI3": "#ff002a",  # 14
    "FAT2": "#0000FF",  # 15
    "MAFB": "#FF00FF",  # 17
}

# 获取 adata.obs 中 leiden_hvg_sub 列的所有唯一值
subset_values = placenta.obs['leiden_hvg_1'].unique().tolist()

fig, curves = plot_genes_along_AP(
    adata=placenta,
    gene_configs=[
        {
            'genes': list(gene_colors.keys()),
            'subset_values': subset_values,  # 使用 leiden_hvg_sub 的所有唯一值
            'use_secondary_y': False,
            'linestyle': '-',
            'n_bins': 40
        },
    ],
    coord_col='AP',
    subset_col='leiden_hvg_sub',  # 改为 leiden_hvg_sub
    smooth=3,
    figsize=(6, 7),
    linewidth=3,
    fill=True,
    show_points=False,
    vertical_lines=vertical_lines,
    vline_width=3,
    show_legend=False,
    x_tick_interval=500,
    secondary_ylim=(0, 1.2),
    gene_colors=gene_colors,
    fill_alpha=0.5,
    xlim=(-300,2800),
    ylim=(0, 1.1)
)

plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure6\along_axis\gene_plot_PLACENTA_AP_dosal_4genes.pdf', dpi=900, bbox_inches='tight')
plt.show()

# 4.Extended Data Fig. 8m

In [ ]:
placenta = sc.read_h5ad(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\analysis\2st_2025_4_9\backup_adata\QC_10_50_2st\\placenta\placenta_for_cellcharter.h5ad")

In [ ]:
import matplotlib.pyplot as plt

cc.pl.enrichment(
    adata,
    group_key='cluster_cellcharter',
    label_key='leiden_hvg_1',
    figsize=(8,6),
    dot_scale=1
)

plt.savefig(
    "enrichment.pdf",
    dpi=300,
    bbox_inches="tight"
)

plt.show()